# DeepTuneIO — Module 10: Performance Visualization

## Purpose

Generate reproducible performance datasets and figures from the normalized `Consolidated Execution` product of a selected campaign.

## Campaign selection

Campaigns are selected with the common DeepTuneIO convention:

- `APPLICATION_POS`: application (1-based index).
- `DATASET_POS`: dataset group inside the selected application.
- `SCENARIO_POS`: scenario inside the selected dataset group.

Current applications:

| APPLICATION_POS | Application |
|---:|---|
| 1 | DeepGalaxy |
| 2 | DLIOv1 |

### DeepGalaxy

`DATASET_POS=1` selects `DG_bw512_f3c5x64`. Scenarios: 1=`lustre_1ost`, 2=`lustre_2ost`, 3=`lustre_4ost`, 4=`nfs_SnGPU_e1`.

### DLIOv1

Dataset positions: 1=`HDF5_64bs`, 2=`NPZ_64bs`, 3=`TFRecord_256kts_64bs`, 4=`TFRecord_1mts_64bs`. All currently use `SCENARIO_POS=1` → `lustre_ost`.

## Analysis options

### `ANALYSIS_SCOPE`

- `article`: reproduce Article 01. Uses `iops_ds_legacy` and the campaign's `article_profile`. If `article_configurations` is `None`, **all valid configurations present in the data are used**. If a list is supplied, only those configurations are retained.
- `all`: general/new DeepTuneIO analysis. Uses `iops_posix` and all valid configurations.

### `ACCESS_MODE`

- `all`: generate one set of outputs for every detected access mode.
- A concrete mode such as `shared`, `shared_reload_shuffle` or `file_per_process` may be selected when needed.

### `IOPS_COLUMN`

- `iops_ds_legacy`: exact Article 01 reproduction.
- `iops_posix`: general/new DeepTuneIO analyses.
- `auto`: schema-priority selection; not recommended for exact historical reproduction.

### `PLOT_PROFILE`

- `bar`: DLIO Fig. 6–9 family.
- `dual`: DeepGalaxy Fig. 11–16 family.
- `all`: generate all supported families.
- `auto`: resolve from `CAMPAIGNS[*][*]["article_profile"]` in Article 01 mode.

### `RUNTIME_COLUMN`

- `auto`: resolve the available runtime metric and record it in the manifest.

## Article 01 policy

For DLIOv1, `article_configurations=None` intentionally means that the current campaign data define the configurations; no node/process filter is imposed. For DeepGalaxy, the published configurations are declared explicitly in `CAMPAIGNS`.

The manifest records the resolved profile, IOPS source, analysis scope and actual configuration filter.


In [15]:
from pathlib import Path
import sys
from importlib.util import spec_from_file_location, module_from_spec

MODULE_DIR = Path("./Module").resolve()
MODULE_FILE = MODULE_DIR / "10_performance_visualization.py"
spec = spec_from_file_location("deeptuneio_viz10", MODULE_FILE)
module10 = module_from_spec(spec); sys.modules[spec.name]=module10; spec.loader.exec_module(module10)
print("Module:", MODULE_FILE)
print("Version:", module10.VERSION)


Module: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\10_Visualization\Performance\10_Performance_Visualization_v1_3_6_1\Module\10_performance_visualization.py
Version: 1.3.6.1


## Select campaign


In [82]:
CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": [
                "lustre_1ost",
                "lustre_2ost",
                "lustre_4ost",
                "nfs_SnGPU_e1"
            ],
            "file_format": ".hdf5",
            "article_profile": "dual",
            "article_iops_column": "iops_ds_legacy",
            "article_configurations": [
                (1, 4),
                (2, 8),
                (4, 16),
                (8, 32),
                (16, 64),
            ],
        },
    },

    "DLIOv1": {
        "HDF5_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".h5",
            "article_profile": "bar",
            "article_iops_column": "iops_ds_legacy",
            "article_configurations": None,
        },

        "NPZ_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".npz",
            "article_profile": "bar",
            "article_iops_column": "iops_posix",
            "article_configurations": None,
        },

        "TFRecord_256kts_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".tfrecords",
            "article_profile": "bar",
            "article_iops_column": "iops_posix",
            "article_configurations": None,
        },

        "TFRecord_1mts_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".tfrecords",
            "article_profile": "bar",
            "article_iops_column": "iops_posix",
            "article_configurations": None,
        },
    },
}

APPLICATION_POS =1
DATASET_POS =1
SCENARIO_POS = 3

APPLICATION = list(CAMPAIGNS)[APPLICATION_POS - 1]
DATASET_GROUP = list(CAMPAIGNS[APPLICATION])[DATASET_POS - 1]
CAMPAIGN_CONFIG = CAMPAIGNS[APPLICATION][DATASET_GROUP]
SCENARIO = CAMPAIGN_CONFIG["scenarios"][SCENARIO_POS - 1]
FILE_FORMAT = CAMPAIGN_CONFIG["file_format"]

ACCESS_MODE = "all"
RUNTIME_COLUMN = "auto"

print("Application   :", APPLICATION)
print("Dataset group :", DATASET_GROUP)
print("Scenario      :", SCENARIO)
print("File format   :", FILE_FORMAT)


Application   : DeepGalaxy
Dataset group : DG_bw512_f3c5x64
Scenario      : lustre_4ost
File format   : .hdf5


In [83]:
# ============================================================
# Reproduction / analysis policy
# ============================================================
#
# "article" -> reproduce Article 01.
# "all"     -> general/new DeepTuneIO analysis.
#

ANALYSIS_SCOPE = "article"

if ANALYSIS_SCOPE == "article":

    IOPS_COLUMN = CAMPAIGN_CONFIG.get(
        "article_iops_column",
        "auto"
    )

    PLOT_PROFILE = CAMPAIGN_CONFIG.get(
        "article_profile",
        "auto"
    )

    ARTICLE_CONFIGURATIONS = CAMPAIGN_CONFIG.get(
        "article_configurations"
    )

else:

    IOPS_COLUMN = "iops_posix"
    PLOT_PROFILE = "all"
    ARTICLE_CONFIGURATIONS = None


print("Analysis scope :", ANALYSIS_SCOPE)
print("IOPS column    :", IOPS_COLUMN)
print("Plot profile   :", PLOT_PROFILE)
print(
    "Configurations :",
    ARTICLE_CONFIGURATIONS
    if ARTICLE_CONFIGURATIONS is not None
    else "ALL VALID CONFIGURATIONS FROM DATA"
)

Analysis scope : article
IOPS column    : iops_ds_legacy
Plot profile   : dual
Configurations : [(1, 4), (2, 8), (4, 16), (8, 32), (16, 64)]


## Build campaign root — same hierarchy as DXT


In [84]:
PROJECT_ROOT=Path(r"F:\\NoteBooks\\CORA_DR_UAB").resolve()
ARTICLE_ID="Article_01_Parallel_IO_Analysis"
ARTICLE_ROOT=PROJECT_ROOT/ARTICLE_ID
CAMPAIGN_ROOT=ARTICLE_ROOT/"Data"/APPLICATION/DATASET_GROUP/SCENARIO
print("Campaign root:",CAMPAIGN_ROOT)
print("Exists:",CAMPAIGN_ROOT.exists())


Campaign root: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\lustre_4ost
Exists: True


## Resolve paths and discover Consolidated Execution


In [85]:
# Resolve paths and preview the exact visualization selection.
scope_args = ["--analysis-scope", ANALYSIS_SCOPE]

if ANALYSIS_SCOPE == "article" and ARTICLE_CONFIGURATIONS is not None:
    configuration_filter = ",".join(f"{nodes}x{processes}" for nodes, processes in ARTICLE_CONFIGURATIONS)
    scope_args += ["--configuration-filter", configuration_filter]

args = module10.parse_args([
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-group", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--file-format", FILE_FORMAT,
    "--event", "JSC24",
    "--module-root", str(MODULE_DIR),
    "--access-mode", ACCESS_MODE,
    "--iops-column", IOPS_COLUMN,
    "--runtime-column", RUNTIME_COLUMN,
    "--profile", PLOT_PROFILE,
] + scope_args)

paths = module10.resolve_campaign_paths(args)
source = module10.discover_input(args, paths)
configs, resolved_profile, resolved_iops, _ = module10.resolve_reproduction_policy(args)

for k, v in paths.items():
    print(f"{k:28s}: {v}")

print("Consolidated CSV:", source)
print("Analysis scope  :", args.analysis_scope)
print("Config filter   :", configs if configs else "ALL VALID CONFIGURATIONS FROM DATA")
print("Resolved profile:", resolved_profile)
print("Resolved IOPS   :", resolved_iops)


campaign_root               : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_4ost
module_root                 : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\10_Visualization\Performance\10_Performance_Visualization_v1_3_6_1\Module
consolidated_execution_dir  : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_4ost\results\Consolidated\Execution
figures_dir                 : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_4ost\results\Figures\Performance
Consolidated CSV: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_4ost\results\Consolidated\Execution\consolidated_execution_DeepGalaxy_DG_bw512_f3c5x64_lustre_4ost_hdf5_JSC24.csv
Analysis scope  : article
Config filter   : [(1, 4), (2, 8), (4, 16), (8, 32), (16, 64)]
Resolved profile: dual
Resolved IOPS   : iops_d

## Execute visualization


In [86]:
scope_args = ["--analysis-scope", ANALYSIS_SCOPE]
if ANALYSIS_SCOPE == "article" and ARTICLE_CONFIGURATIONS is not None:
    configuration_filter = ",".join(f"{nodes}x{processes}" for nodes, processes in ARTICLE_CONFIGURATIONS)
    scope_args += ["--configuration-filter", configuration_filter]

rc = module10.main([
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-group", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--file-format", FILE_FORMAT,
    "--event", "JSC24",
    "--module-root", str(MODULE_DIR),
    "--access-mode", ACCESS_MODE,
    "--iops-column", IOPS_COLUMN,
    "--runtime-column", RUNTIME_COLUMN,
    "--profile", PLOT_PROFILE,
] + scope_args)

print("Return code:", rc)


DeepTuneIO — Module 10: Performance Visualization
Campaign root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_4ost
Source        : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_4ost\results\Consolidated\Execution\consolidated_execution_DeepGalaxy_DG_bw512_f3c5x64_lustre_4ost_hdf5_JSC24.csv
Application   : DeepGalaxy
Dataset group : DG_bw512_f3c5x64
Scenario      : lustre_4ost
Profile       : dual
Analysis scope: article
Config filter : [(1, 4), (2, 8), (4, 16), (8, 32), (16, 64)]
IOPS request  : iops_ds_legacy
Metric sources: {'io_time': 'io_time_s', 'runtime': 'perf__Run_Time_s', 'bandwidth': 'bandwidth_mib_s', 'iops': 'iops_ds_legacy', 'memory': 'memory_utilized_bytes', 'access_mode': 'access_pattern_detected', 'transfer_size': 'indexer__transfer_size_label'}
Access modes  : ['shared', 'shared_reload_shuffle']
[plot-style] application=DeepGalaxy scenario=lustre_4ost file_format=.hd

## Output identification and provenance

Each canonical grouped CSV includes campaign context columns (`application`, `dataset_group`, `scenario`, `filesystem`, `file_format`, `access_mode`) in addition to the grouped configuration and metric statistics. This makes the product interpretable even when copied outside the original campaign directory.

Legacy filenames are retained temporarily only for compatibility with downstream modules that still expect `performance_grouped_<mode>.csv`. The canonical self-identifying filename is the preferred product for CORA/RDR deposition and future processing.

## Inspect generated files


In [21]:
FIG_DIR=CAMPAIGN_ROOT/"results"/"Figures"/"Performance"
for p in sorted(FIG_DIR.rglob("*")) if FIG_DIR.exists() else []:
    if p.is_file(): print(" -",p.relative_to(FIG_DIR))


 - Data\performance_grouped_dliov1_hdf5_64bs_lustre_ost_h5_shared.csv
 - Data\performance_grouped_shared.csv
 - performance_visualization_manifest.json
 - shared\01_execution_time_breakdown.pdf
 - shared\01_execution_time_breakdown.png
 - shared\02_normalized_execution_time_breakdown.pdf
 - shared\02_normalized_execution_time_breakdown.png
 - shared\03_bandwidth_per_configuration.pdf
 - shared\03_bandwidth_per_configuration.png
 - shared\04_iops_per_configuration.pdf
 - shared\04_iops_per_configuration.png
